# measure accelerations in poster children

## author:
- **David W. Hogg** (NYU) (Flatiron) (MPIA)
- **Abby P. Shaum** (Columbia)

## bugs:
- Needs to be audited for units (days vs seconds).
- Ought to do periodograms (long periods only) to see if the weird signals are at one year.
- Ought to visualize the clock as well as the o-c phase shift.

## project:
- See if I can measure an acceleration in one of our precise *Kepler* clocks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from astropy.timeseries import LombScargle as LS

In [ ]:
# all this to import the KeplerClocks code

from pathlib import Path
import sys
target_dir = Path.cwd() / "../../KeplerClocks/py"
sys.path.append(str(target_dir.resolve()))
import clocks as kc

In [ ]:
seconds_per_day = 86_400
rng = np.random.default_rng(17)

In [ ]:
# choose a poster child
# kicid, expected_period = "KIC007917485", 840. # SJM 840-day planet host
kicid, expected_period = "KIC006780873", 9.16 # SJM 9.16-day binary companion

In [ ]:
# get best clocks in this child
clock_table = kc.best_clocks_in_star(kicid).to_pandas()

In [ ]:
print(clock_table)
clock_table['star_id'] = kicid

In [ ]:
def get_ominusc_for_one_clock(clock, nchunk=32, nboot=31, njack=7):

    # get data
    ts, ys, errs, df, dt = kc.get_kepler_data(clock["star_id"])
    ivars = errs ** -2

    # fit and get residuals; project onto derivative
    X, ms, pars = kc.fourier_wls_fit(clock["angular_frequency"], clock["fourier_series_degree"],
                                     ts, ys, ivars)
    resids = ys - X @ pars
    deriv_pars = kc.take_derivative_wrt_phase(pars, ms)
    derivs = clock['angular_frequency'] * X @ deriv_pars # dflux / dt

    # average o-c in chunks
    chunk_ts, chunk_advances = np.zeros(nchunk), np.zeros(nchunk)
    chunk_ivars, chunk_boot_vars, chunk_jack_vars = np.zeros(nchunk), np.zeros(nchunk), np.zeros(nchunk)
    for chunk in range(nchunk):
        a, b = np.percentile(ts, [100 * chunk / nchunk, 100 * (chunk + 1) / nchunk])
        inchunk = (ts >= a) & (ts <= b)
        ninchunk = np.sum(inchunk)
        inchunk = np.arange(len(ts))[inchunk]
        assert len(inchunk) == ninchunk
        chunk_ts[chunk] = np.sum(ivars[inchunk] * ts[inchunk]) \
                        / np.sum(ivars[inchunk])
        chunk_advances[chunk] = np.sum(ivars[inchunk] * resids[inchunk] * derivs[inchunk]) \
                              / np.sum(ivars[inchunk] * derivs[inchunk] * derivs[inchunk])
        chunk_ivars[chunk]    = np.sum(ivars[inchunk] * derivs[inchunk] * derivs[inchunk])
        boots = np.zeros(nboot) + np.nan
        for i in range(nboot):
            thisidx = rng.choice(inchunk, size=ninchunk)
            boots[i] = np.sum(ivars[thisidx] * resids[thisidx] * derivs[thisidx]) \
                     / np.sum(ivars[thisidx] * derivs[thisidx] * derivs[thisidx])
        chunk_boot_vars[chunk] = np.var(boots, ddof=1)
        jacknum = (np.arange(ninchunk) * njack) // ninchunk
        jacks = np.zeros(njack) + np.nan
        for i in range(njack):
            thisidx = inchunk[jacknum != i]
            jacks[i] = np.sum(ivars[thisidx] * resids[thisidx] * derivs[thisidx]) \
                     / np.sum(ivars[thisidx] * derivs[thisidx] * derivs[thisidx])
        chunk_jack_vars[chunk] = (1. / (njack * (njack - 1))) * np.sum((jacks - np.mean(jacks)) ** 2)
    return ts, ys, ivars, resids, derivs, chunk_ts, chunk_advances, chunk_ivars, chunk_boot_vars, chunk_jack_vars

In [ ]:
# is there a trend in the residuals?

def get_and_plot_one_ominusc(clock):
    f = plt.figure(figsize=(9, 3))
    plt.axhline(0, c="k", lw=0.5)
    for nchunk in [32, ]:
        ts, ys, ivars, resids, derivs, chunk_ts, advances, _, boot_vars, jack_vars = get_ominusc_for_one_clock(clock, nchunk=nchunk)
        plt.step(chunk_ts, seconds_per_day * advances, c="k", where="mid")
        plt.errorbar(chunk_ts, seconds_per_day * advances,
                 yerr = seconds_per_day * np.sqrt(jack_vars),
                 marker=".", color="k", linestyle="none", label="jackknife uncertainties")
    plt.legend()
    plt.xlabel("Barycentric time BJD [d]")
    plt.ylabel("advance $O-C$ [s]")
    plt.title(f"timing offsets of clock in {clock["star_id"]} with period {2. * np.pi / clock["angular_frequency"]:.5f} d")
    plt.show()
    frequency, numerator   = LS(ts, resids * ivars * derivs, normalization="psd").autopower()
    frequency, denominator = LS(ts, derivs * ivars * derivs, normalization="psd").autopower()
    f2 = plt.figure(figsize=(9, 3))
    if expected_period is not None:
        plt.axvline(expected_period, color="r")
    plt.plot(1. / frequency, numerator, "k-")
    plt.semilogx()
    plt.xlim(3., 1600.)
    foo = np.max(numerator[1. / frequency > 3])
    plt.ylim(-0.1 * foo, 1.1 * foo)
    plt.show()
    return f

In [ ]:
print(clock_table["star_id"])
for i, clock in clock_table.iterrows():
    
    print(clock)
    print(clock["star_id"])
    get_and_plot_one_ominusc(clock)